# 🧠 EX59: Multi-Task Modes

YOLO11 shares Backbone+Neck across tasks; only prediction Heads differ.

| Task | Suffix | Output attr | Example use |
|------|--------|-------------|-------------|
| Detect | `.pt` | `.boxes` | Vehicle counting |
| Segment | `-seg.pt` | `.masks` | Medical area measurement |
| Classify | `-cls.pt` | `.probs` | Defect classification |
| Pose | `-pose.pt` | `.keypoints` | Sports biomechanics |
| OBB | `-obb.pt` | `.obb` | Aerial imagery |

**Cost order:** Classify < Detect < Pose < OBB ≈ Segment

## 🔗 Links
- [[EX58_Model_Export_TH]] | [[EX60_Streaming_Inference_TH]]


In [ ]:
# Back up or checkpoint this section of code before starting to modify the large file.
import gc
import cv2, numpy as np, torch
import matplotlib.pyplot as plt
from solution import get_yolo_model_by_task
%matplotlib inline

device = "0" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Device: {device}")

dummy = np.zeros((480, 640, 3), dtype=np.uint8)
cv2.rectangle(dummy, (100,100), (400,350), (200,200,200), -1)

tasks = {
    "detect":   lambda r: f"boxes={len(r.boxes)}",
    "segment":  lambda r: f"masks={'present' if r.masks else 'None'}",
    "classify": lambda r: f"top1={r.probs.top1} conf={r.probs.top1conf.item():.3f}",
    "pose":     lambda r: f"kp_shape={r.keypoints.xy.shape if r.keypoints else 'None'}",
}

print("\n--- AUDIT & INSPECTION START ---")
results_map = {}
for task, summarize in tasks.items():
    print(f"\n[{task.upper()}]")
    model = get_yolo_model_by_task(task)
    res = model.predict(source=dummy, verbose=False)
    results_map[task] = res[0]
    print(f"  Summary: {summarize(res[0])}")
    if task=="detect" and res[0].boxes:
        print(f"  boxes.xyxy: {res[0].boxes.xyxy.shape}")
    elif task=="segment" and res[0].masks:
        print(f"  masks.data: {res[0].masks.data.shape}")
    elif task=="classify" and res[0].probs:
        print(f"  probs.data: {res[0].probs.data.shape}")
    elif task=="pose" and res[0].keypoints:
        print(f"  kp.xy: {res[0].keypoints.xy.shape}")
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
print("\n--- AUDIT & INSPECTION END ---")

fig, axes = plt.subplots(1, len(results_map), figsize=(4*len(results_map),4))
for ax,(task,result) in zip(axes, results_map.items()):
    ax.imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
    ax.set_title(task.capitalize()); ax.axis("off")
plt.suptitle("YOLO11 Multi-Task Results"); plt.tight_layout(); plt.show()
